In [2]:
import pandas as pd
import numpy as np

In [1]:
import pandas as pd
import numpy as np

# Load the .raw file
file_path = "/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/BDR_AD_APOE_DAB1.raw"
df = pd.read_csv(file_path, sep=" ", header=0)

# Column names for rs429358 and rs7412
rs429358_col = "chr19:44908684:T:C_C"  # dosage of C, risk allele = T → flip
rs7412_col   = "chr19:44908822:C:T_T"  # dosage of T, risk allele = C → flip

# Flip function: ALT dosage → REF dosage
def flip_dosage(series):
    return 2 - series

# Apply flipping and round to nearest integer
df["rs429358_dosage"] = flip_dosage(df[rs429358_col]).round().astype(int)
df["rs7412_dosage"]   = flip_dosage(df[rs7412_col]).round().astype(int)

# Assign APOE genotype based on risk-allele dosages
def assign_apoe(a, b):
    if (a, b) == (0, 0):
        return "e2e2"
    elif (a, b) in [(0, 1), (1, 0)]:
        return "e2e3"
    elif (a, b) == (0, 2):
        return "e3e3"
    elif (a, b) in [(1, 1)]:
        return "e2e4"
    elif (a, b) in [(1, 2), (2, 1)]:
        return "e3e4"
    elif (a, b) == (2, 2):
        return "e4e4"
    else:
        return np.nan

df["APOE_dosage"] = [assign_apoe(a, b) for a, b in zip(df["rs429358_dosage"], df["rs7412_dosage"])]

# Function to count number of e4 alleles
def count_e4_alleles(genotype):
    if pd.isna(genotype):
        return 0
    return int(genotype.count('e4'))

# Apply the function
df["APOE_e4_count"] = df["APOE_dosage"].apply(count_e4_alleles)

df = df.rename(columns={"chr1:57182042:C:T_T": "DAB1"})
df = df.rename(columns={"APOE_e4_count": "APOE"})

ml_AD_APOE_DAB1 = df[["PHENOTYPE", "DAB1", "APOE"]]

ml_AD_APOE_DAB1["PHENOTYPE"] = ml_AD_APOE_DAB1["PHENOTYPE"].map({1: 0, 2: 1})

ml_AD_APOE_DAB1_full = df[["FID","PHENOTYPE", "DAB1", "APOE"]]
ml_AD_APOE_DAB1_full["PHENOTYPE"] = ml_AD_APOE_DAB1_full["PHENOTYPE"].map({1: 0, 2: 1})

print(ml_AD_APOE_DAB1)
print(ml_AD_APOE_DAB1_full)
# Save to CSV
ml_AD_APOE_DAB1.to_csv("/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/ml_AD_APOE_DAB1.csv", index=False)
ml_AD_APOE_DAB1_full.to_csv("/scratch/c.c2029098/dementia_ml_project/data/processed/ml_data/ml_AD_APOE_DAB1_full.csv", index=False)

     PHENOTYPE  DAB1  APOE
0            0     0     2
1            1     1     0
2            1     0     1
3            0     0     1
4            0     0     2
..         ...   ...   ...
515          0     0     1
516          0     0     2
517          0     0     2
518          0     0     2
519          1     0     0

[520 rows x 3 columns]
                           FID  PHENOTYPE  DAB1  APOE
0    201023670019_R01C01_BB001          0     0     2
1    201023670019_R01C02_BB013          1     1     0
2    201023670019_R02C01_BB007          1     0     1
3    201023670019_R02C02_BB019          0     0     1
4    201023670019_R03C01_BB002          0     0     2
..                         ...        ...   ...   ...
515  202136020022_R10C02_BK250          0     0     1
516  202136020022_R11C01_BN090          0     0     2
517  202136020022_R11C02_BK245          0     0     2
518  202136020022_R12C01_BK238          0     0     2
519  202136020022_R12C02_BK251          1     0     0

[52

/tmp/ipykernel_23766/4213451893.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_AD_APOE_DAB1["PHENOTYPE"] = ml_AD_APOE_DAB1["PHENOTYPE"].map({1: 0, 2: 1})
/tmp/ipykernel_23766/4213451893.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_AD_APOE_DAB1_full["PHENOTYPE"] = ml_AD_APOE_DAB1_full["PHENOTYPE"].map({1: 0, 2: 1})
